# Node Impact Analysis

This notebook analyzes the impact of a disruption at a specific node (plant/location) in the supply chain.

**Parameters:**
- `IMPACTED_NODE`: Plant code to analyze (default: single plant, can be comma-separated)
- `IMPACTED_PRODUCTS`: Material numbers to analyze (default: ALL)
- `IMPACT_DATE`: Date of impact (YYYYMMDD)
- `IMPACT_DURATION`: Duration of impact in days

**Output Tables:**
1. Current inventory at impacted location(s) - from MARD/MARC/MBEW
2. Current inventory at all other locations - from MARD/MARC/MBEW
3. Planned goods receipts (inbound) - from PLAF (production + stock transfers)
4. Planned goods issues (outbound) - from PLAF (stock transfers) + VBAP/VBEP (customer orders)
5. Transportation lanes (T-Lanes) - from SAPAPO_TR/SAPAPO_TRM

In [ ]:
# =============================================================================
# CONFIGURATION PARAMETERS
# =============================================================================

# --- Database Connection ---
dbutils.widgets.text("CATALOG", "sample_synthetic_sap", "Catalog Name")
dbutils.widgets.text("SCHEMA", "sap", "Schema Name")

# --- Impact Parameters ---
dbutils.widgets.text("IMPACTED_NODE", "1000", "Impacted Node (plant code)")
dbutils.widgets.text("IMPACTED_PRODUCTS", "ALL", "Impacted Products (ALL or comma-separated)")
dbutils.widgets.text("IMPACT_DATE", "20250615", "Impact Date (YYYYMMDD)")
dbutils.widgets.text("IMPACT_DURATION", "30", "Impact Duration (days)")

# --- Get Parameter Values ---
CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
IMPACTED_NODE = dbutils.widgets.get("IMPACTED_NODE")
IMPACTED_PRODUCTS = dbutils.widgets.get("IMPACTED_PRODUCTS")
IMPACT_DATE = dbutils.widgets.get("IMPACT_DATE")
IMPACT_DURATION = int(dbutils.widgets.get("IMPACT_DURATION"))

import pandas as pd
from datetime import datetime, timedelta

# Parse dates
impact_date_dt = datetime.strptime(IMPACT_DATE, '%Y%m%d')
end_date_dt = impact_date_dt + timedelta(days=IMPACT_DURATION)
IMPACT_DATE_STR = impact_date_dt.strftime('%Y%m%d')
END_DATE_STR = end_date_dt.strftime('%Y%m%d')

# Parse node list
IMPACTED_NODES_LIST = [n.strip() for n in IMPACTED_NODE.split(',')]
IMPACTED_NODES_SQL = "','".join(IMPACTED_NODES_LIST)

# Parse product filter
if IMPACTED_PRODUCTS.upper() == 'ALL':
    PRODUCT_FILTER = "1=1"  # No filter
    PRODUCT_FILTER_DISPLAY = "ALL"
else:
    IMPACTED_PRODUCTS_LIST = [p.strip() for p in IMPACTED_PRODUCTS.split(',')]
    IMPACTED_PRODUCTS_SQL = "','" .join(IMPACTED_PRODUCTS_LIST)
    PRODUCT_FILTER = f"MATNR IN ('{IMPACTED_PRODUCTS_SQL}')"
    PRODUCT_FILTER_DISPLAY = IMPACTED_PRODUCTS

print("=" * 70)
print("NODE IMPACT ANALYSIS - CONFIGURATION")
print("=" * 70)
print(f"Database:           {CATALOG}.{SCHEMA}")
print(f"Impacted Node(s):   {IMPACTED_NODE}")
print(f"Impacted Products:  {PRODUCT_FILTER_DISPLAY}")
print(f"Impact Date:        {IMPACT_DATE_STR}")
print(f"Impact Duration:    {IMPACT_DURATION} days")
print(f"Analysis Period:    {IMPACT_DATE_STR} to {END_DATE_STR}")
print("=" * 70)

---
## 1. Current Inventory at Impacted Location(s)

Inventory currently held at the impacted node(s).

In [ ]:
# Current Inventory at Impacted Location(s)
inventory_impacted = spark.sql(f"""
    SELECT 
        d.WERKS as Location,
        l.CITY as Location_Name,
        d.LGORT as Storage_Location,
        d.MATNR as Product_Code,
        t.MAKTX as Product_Description,
        CAST(d.LABST AS DOUBLE) as Quantity,
        COALESCE(CAST(c.EISBE AS DOUBLE), 0) as Safety_Stock_Qty,
        COALESCE(b.STPRS, 0) as Standard_Price,
        COALESCE(b.PEINH, 1) as Price_Unit,
        ROUND(CAST(d.LABST AS DOUBLE) * COALESCE(b.STPRS, 0) / COALESCE(b.PEINH, 1), 2) as Current_Value_GBP
    FROM {CATALOG}.{SCHEMA}.mard d
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON d.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.marc c ON d.MATNR = c.MATNR AND d.WERKS = c.WERKS
    LEFT JOIN {CATALOG}.{SCHEMA}.mbew b ON d.MATNR = b.MATNR AND d.WERKS = b.BWKEY
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l ON d.WERKS = l.LOCNO
    WHERE d.WERKS IN ('{IMPACTED_NODES_SQL}')
    AND {PRODUCT_FILTER.replace('MATNR', 'd.MATNR')}
    AND CAST(d.LABST AS DOUBLE) > 0
    ORDER BY d.WERKS, d.LGORT, d.MATNR
""")

df_inv_impacted = inventory_impacted.toPandas()

print("CURRENT INVENTORY AT IMPACTED LOCATION(S)")
print(f"Location(s): {IMPACTED_NODE}")
print("=" * 70)

if len(df_inv_impacted) > 0:
    print(f"Total Products:      {df_inv_impacted['Product_Code'].nunique()}")
    print(f"Total Quantity:      {df_inv_impacted['Quantity'].sum():,.0f} units")
    print(f"Total Safety Stock:  {df_inv_impacted['Safety_Stock_Qty'].sum():,.0f} units")
    print(f"Total Value:         £{df_inv_impacted['Current_Value_GBP'].sum():,.2f}")
else:
    print("No inventory found at impacted location(s).")

print("=" * 70)
display(inventory_impacted)

---
## 2. Current Inventory at Other Locations

Inventory at all locations excluding the impacted node(s) - potential alternative supply sources.

In [ ]:
# Current Inventory at Other Locations (excluding impacted)
inventory_other = spark.sql(f"""
    SELECT 
        d.WERKS as Location,
        l.CITY as Location_Name,
        d.LGORT as Storage_Location,
        d.MATNR as Product_Code,
        t.MAKTX as Product_Description,
        CAST(d.LABST AS DOUBLE) as Quantity,
        COALESCE(CAST(c.EISBE AS DOUBLE), 0) as Safety_Stock_Qty,
        COALESCE(b.STPRS, 0) as Standard_Price,
        COALESCE(b.PEINH, 1) as Price_Unit,
        ROUND(CAST(d.LABST AS DOUBLE) * COALESCE(b.STPRS, 0) / COALESCE(b.PEINH, 1), 2) as Current_Value_GBP
    FROM {CATALOG}.{SCHEMA}.mard d
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON d.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.marc c ON d.MATNR = c.MATNR AND d.WERKS = c.WERKS
    LEFT JOIN {CATALOG}.{SCHEMA}.mbew b ON d.MATNR = b.MATNR AND d.WERKS = b.BWKEY
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l ON d.WERKS = l.LOCNO
    WHERE d.WERKS NOT IN ('{IMPACTED_NODES_SQL}')
    AND {PRODUCT_FILTER.replace('MATNR', 'd.MATNR')}
    AND CAST(d.LABST AS DOUBLE) > 0
    ORDER BY d.WERKS, d.LGORT, d.MATNR
""")

df_inv_other = inventory_other.toPandas()

print("CURRENT INVENTORY AT OTHER LOCATIONS")
print(f"Excluding: {IMPACTED_NODE}")
print("=" * 70)

if len(df_inv_other) > 0:
    # Summary by location
    loc_summary = df_inv_other.groupby(['Location', 'Location_Name']).agg({
        'Product_Code': 'nunique',
        'Quantity': 'sum',
        'Safety_Stock_Qty': 'sum',
        'Current_Value_GBP': 'sum'
    }).reset_index()
    
    for _, row in loc_summary.iterrows():
        print(f"Plant {row['Location']} ({row['Location_Name']})")
        print(f"  Products: {row['Product_Code']}  |  Qty: {row['Quantity']:,.0f}  |  Value: £{row['Current_Value_GBP']:,.2f}")
    
    print("=" * 70)
    print(f"TOTAL: {df_inv_other['Product_Code'].nunique()} products  |  {df_inv_other['Quantity'].sum():,.0f} units  |  £{df_inv_other['Current_Value_GBP'].sum():,.2f}")
else:
    print("No inventory found at other locations.")

print()
display(inventory_other)

---
## 3. Planned Goods Receipts (Inbound to Impacted Location)

Planned orders scheduled to be received at the impacted location during the impact period.
Includes both production orders (BESKZ='E') and stock transfers (BESKZ='U') from PLAF.

In [ ]:
# Planned Goods Receipts - from PLAF (Planned Orders)
# Includes production (BESKZ='E') and stock transfers (BESKZ='U')
planned_receipts = spark.sql(f"""
    SELECT 
        p.WERKS as Receiving_Location,
        l_recv.CITY as Receiving_Location_Name,
        CASE 
            WHEN p.BESKZ = 'E' THEN 'PRODUCTION'
            WHEN p.BESKZ = 'U' THEN p.LWERK
            ELSE 'OTHER'
        END as Ship_From,
        CASE 
            WHEN p.BESKZ = 'U' THEN l_ship.CITY
            ELSE 'In-house Production'
        END as Ship_From_Name,
        p.MATNR as Product_Code,
        t.MAKTX as Product_Description,
        p.PEDTR as Planned_Receipt_Date,
        CAST(p.GSMNG AS DOUBLE) as Quantity,
        p.MEINS as Unit,
        p.PLNUM as Order_Number,
        CASE 
            WHEN p.BESKZ = 'E' THEN 'Production'
            WHEN p.BESKZ = 'U' THEN 'Stock Transfer'
            ELSE p.BESKZ
        END as Order_Type
    FROM {CATALOG}.{SCHEMA}.plaf p
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON p.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l_recv ON p.WERKS = l_recv.LOCNO
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l_ship ON p.LWERK = l_ship.LOCNO
    WHERE p.WERKS IN ('{IMPACTED_NODES_SQL}')
    AND p.PEDTR >= '{IMPACT_DATE_STR}'
    AND p.PEDTR <= '{END_DATE_STR}'
    AND {PRODUCT_FILTER.replace('MATNR', 'p.MATNR')}
    ORDER BY p.PEDTR, p.WERKS, p.MATNR
""")

df_receipts = planned_receipts.toPandas()

print("PLANNED GOODS RECEIPTS - INBOUND TO IMPACTED LOCATION")
print(f"Receiving Location(s): {IMPACTED_NODE}")
print(f"Period: {IMPACT_DATE_STR} to {END_DATE_STR}")
print("=" * 70)

if len(df_receipts) > 0:
    print(f"Total Planned Orders: {len(df_receipts)}")
    print(f"Total Products:       {df_receipts['Product_Code'].nunique()}")
    print(f"Total Quantity:       {df_receipts['Quantity'].sum():,.0f} units")
    
    # Summary by order type
    type_summary = df_receipts.groupby('Order_Type').agg({
        'Order_Number': 'count',
        'Quantity': 'sum'
    }).reset_index()
    print("\nBy Order Type:")
    for _, row in type_summary.iterrows():
        print(f"  {row['Order_Type']}: {row['Order_Number']} orders, {row['Quantity']:,.0f} units")
    
    # Summary by date
    print("\nBy Planned Date:")
    date_summary = df_receipts.groupby('Planned_Receipt_Date').agg({
        'Order_Number': 'count',
        'Quantity': 'sum'
    }).reset_index()
    for _, row in date_summary.head(10).iterrows():
        print(f"  {row['Planned_Receipt_Date']}: {row['Order_Number']} orders, {row['Quantity']:,.0f} units")
    if len(date_summary) > 10:
        print(f"  ... and {len(date_summary) - 10} more dates")
else:
    print("No planned receipts found for the impacted location(s) in this period.")

print("=" * 70)
display(planned_receipts)

---
## 4. Planned Goods Issues (Outbound from Impacted Location)

Planned outbound movements from the impacted location during the impact period.
Includes:
- Stock transfers to other nodes (from PLAF where FLIEF = impacted location)
- Customer orders (from VBAP/VBEP - open sales orders)

In [ ]:
# Planned Goods Issues - Combined view of transfers and customer orders
# 1. Stock transfers FROM impacted location (PLAF BESKZ='U' where LWERK = impacted)
# 2. Customer orders shipping from impacted location (VBAP/VBEP)

# Part 1: Stock Transfers (from PLAF)
planned_transfers_out = spark.sql(f"""
    SELECT 
        p.LWERK as Ship_From_Location,
        l_from.CITY as Ship_From_Name,
        p.WERKS as Ship_To_Location,
        l_to.CITY as Ship_To_Name,
        'INTERNAL' as Customer_Type,
        p.MATNR as Product_Code,
        t.MAKTX as Product_Description,
        p.PSTTR as Planned_Issue_Date,
        CAST(p.GSMNG AS DOUBLE) as Quantity,
        p.MEINS as Unit,
        p.PLNUM as Order_Number,
        'Stock Transfer' as Order_Type
    FROM {CATALOG}.{SCHEMA}.plaf p
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON p.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l_from ON p.LWERK = l_from.LOCNO
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l_to ON p.WERKS = l_to.LOCNO
    WHERE p.BESKZ = 'U'
    AND p.LWERK IN ('{IMPACTED_NODES_SQL}')
    AND p.PSTTR >= '{IMPACT_DATE_STR}'
    AND p.PSTTR <= '{END_DATE_STR}'
    AND {PRODUCT_FILTER.replace('MATNR', 'p.MATNR')}
""")

# Part 2: Customer Orders (from VBAP/VBEP)
planned_customer_orders = spark.sql(f"""
    WITH delivered_orders AS (
        SELECT DISTINCT f.VBELN
        FROM {CATALOG}.{SCHEMA}.vbfa f
        INNER JOIN {CATALOG}.{SCHEMA}.likp d ON f.VBELN_N = d.VBELN
    )
    SELECT 
        p.WERKS as Ship_From_Location,
        l.CITY as Ship_From_Name,
        o.KUNNR as Ship_To_Location,
        c.NAME1 as Ship_To_Name,
        'CUSTOMER' as Customer_Type,
        p.MATNR as Product_Code,
        t.MAKTX as Product_Description,
        e.EDATU as Planned_Issue_Date,
        CAST(e.BMENG AS DOUBLE) as Quantity,
        p.VRKME as Unit,
        o.VBELN as Order_Number,
        'Customer Order' as Order_Type
    FROM {CATALOG}.{SCHEMA}.vbak o
    INNER JOIN {CATALOG}.{SCHEMA}.vbap p ON o.VBELN = p.VBELN
    INNER JOIN {CATALOG}.{SCHEMA}.vbep e ON p.VBELN = e.VBELN AND p.POSNR = e.POSNR
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON p.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.kna1 c ON o.KUNNR = c.KUNNR
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l ON p.WERKS = l.LOCNO
    WHERE p.WERKS IN ('{IMPACTED_NODES_SQL}')
    AND o.VBELN NOT IN (SELECT VBELN FROM delivered_orders)
    AND e.EDATU >= '{IMPACT_DATE_STR}'
    AND e.EDATU <= '{END_DATE_STR}'
    AND {PRODUCT_FILTER.replace('MATNR', 'p.MATNR')}
""")

# Combine both
planned_issues = planned_transfers_out.union(planned_customer_orders)
df_issues = planned_issues.toPandas()

print("PLANNED GOODS ISSUES - OUTBOUND FROM IMPACTED LOCATION")
print(f"Ship From Location(s): {IMPACTED_NODE}")
print(f"Period: {IMPACT_DATE_STR} to {END_DATE_STR}")
print("=" * 70)

if len(df_issues) > 0:
    print(f"Total Planned Issues: {len(df_issues)}")
    print(f"Total Products:       {df_issues['Product_Code'].nunique()}")
    print(f"Total Quantity:       {df_issues['Quantity'].sum():,.0f} units")
    
    # Summary by order type
    type_summary = df_issues.groupby('Order_Type').agg({
        'Order_Number': 'nunique',
        'Quantity': 'sum'
    }).reset_index()
    print("\nBy Order Type:")
    for _, row in type_summary.iterrows():
        print(f"  {row['Order_Type']}: {row['Order_Number']} orders, {row['Quantity']:,.0f} units")
    
    # Summary by customer type
    cust_summary = df_issues.groupby('Customer_Type').agg({
        'Ship_To_Location': 'nunique',
        'Quantity': 'sum'
    }).reset_index()
    print("\nBy Destination Type:")
    for _, row in cust_summary.iterrows():
        dest_type = "Internal Plants" if row['Customer_Type'] == 'INTERNAL' else "External Customers"
        print(f"  {dest_type}: {row['Ship_To_Location']} destinations, {row['Quantity']:,.0f} units")
    
    # Summary by date
    print("\nBy Planned Date (first 10):")
    date_summary = df_issues.groupby('Planned_Issue_Date').agg({
        'Order_Number': 'nunique',
        'Quantity': 'sum'
    }).reset_index().sort_values('Planned_Issue_Date')
    for _, row in date_summary.head(10).iterrows():
        print(f"  {row['Planned_Issue_Date']}: {row['Order_Number']} orders, {row['Quantity']:,.0f} units")
    if len(date_summary) > 10:
        print(f"  ... and {len(date_summary) - 10} more dates")
else:
    print("No planned issues found from the impacted location(s) in this period.")

print("=" * 70)
display(planned_issues)

---
## 5. Transportation Lanes (T-Lanes)

Transportation lanes connected to the impacted location(s) - showing available routes for rerouting or alternative supply.

In [ ]:
# Transportation Lanes connected to impacted location(s)
# From SAPAPO_TR (lanes) and SAPAPO_TRM (modes)
tlanes = spark.sql(f"""
    SELECT 
        tr.TRLID as Lane_ID,
        tr.TRNAME as Lane_Name,
        tr.LOCFR as From_Location,
        l_from.CITY as From_City,
        tr.LOCTO as To_Location,
        l_to.CITY as To_City,
        CASE 
            WHEN tr.LOCFR IN ('{IMPACTED_NODES_SQL}') THEN 'OUTBOUND'
            WHEN tr.LOCTO IN ('{IMPACTED_NODES_SQL}') THEN 'INBOUND'
        END as Direction,
        trm.TRMID as Mode,
        CAST(trm.TRAESSION AS INT) as Transit_Days,
        CAST(trm.TRACOST AS DOUBLE) as Cost_Per_Unit,
        CASE WHEN trm.PRIFLAG = 'X' THEN 'Primary' ELSE 'Alternate' END as Priority
    FROM {CATALOG}.{SCHEMA}.sapapo_tr tr
    INNER JOIN {CATALOG}.{SCHEMA}.sapapo_trm trm ON tr.TRLID = trm.TRLID
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l_from ON tr.LOCFR = l_from.LOCNO
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l_to ON tr.LOCTO = l_to.LOCNO
    WHERE tr.LOCFR IN ('{IMPACTED_NODES_SQL}') 
       OR tr.LOCTO IN ('{IMPACTED_NODES_SQL}')
    ORDER BY Direction, tr.TRLID, trm.TRAESSION
""")

df_tlanes = tlanes.toPandas()

print("TRANSPORTATION LANES CONNECTED TO IMPACTED LOCATION")
print(f"Location(s): {IMPACTED_NODE}")
print("=" * 70)

if len(df_tlanes) > 0:
    # Summary by direction
    inbound = df_tlanes[df_tlanes['Direction'] == 'INBOUND']
    outbound = df_tlanes[df_tlanes['Direction'] == 'OUTBOUND']
    
    print(f"Inbound Lanes (TO impacted):    {inbound['Lane_ID'].nunique()}")
    print(f"Outbound Lanes (FROM impacted): {outbound['Lane_ID'].nunique()}")
    print(f"Total Lane Options:             {len(df_tlanes)}")
    
    # Summary by mode
    print("\nBy Transport Mode:")
    mode_summary = df_tlanes.groupby('Mode').agg({
        'Lane_ID': 'nunique',
        'Transit_Days': 'mean',
        'Cost_Per_Unit': 'mean'
    }).reset_index()
    for _, row in mode_summary.iterrows():
        print(f"  {row['Mode']}: {row['Lane_ID']} lanes, avg {row['Transit_Days']:.1f} days, avg £{row['Cost_Per_Unit']:.2f}/unit")
    
    # Primary vs Alternate
    print("\nBy Priority:")
    pri_summary = df_tlanes.groupby('Priority').agg({'Lane_ID': 'nunique'}).reset_index()
    for _, row in pri_summary.iterrows():
        print(f"  {row['Priority']}: {row['Lane_ID']} lanes")
else:
    print("No transportation lanes found for the impacted location(s).")

print("=" * 70)
display(tlanes)

---
## 6. Summary

In [ ]:
# Summary
print("\n" + "=" * 70)
print("NODE IMPACT ANALYSIS - SUMMARY")
print("=" * 70)
print(f"Impacted Node(s):    {IMPACTED_NODE}")
print(f"Impacted Products:   {PRODUCT_FILTER_DISPLAY}")
print(f"Impact Period:       {IMPACT_DATE_STR} to {END_DATE_STR} ({IMPACT_DURATION} days)")
print("=" * 70)

print(f"\n{'INVENTORY AT IMPACTED LOCATION':<40}")
print(f"  Products:              {df_inv_impacted['Product_Code'].nunique() if len(df_inv_impacted) > 0 else 0:>15,}")
print(f"  Quantity:              {df_inv_impacted['Quantity'].sum() if len(df_inv_impacted) > 0 else 0:>15,.0f}")
print(f"  Value:                 £{df_inv_impacted['Current_Value_GBP'].sum() if len(df_inv_impacted) > 0 else 0:>14,.2f}")

print(f"\n{'INVENTORY AT OTHER LOCATIONS':<40}")
print(f"  Locations:             {df_inv_other['Location'].nunique() if len(df_inv_other) > 0 else 0:>15,}")
print(f"  Products:              {df_inv_other['Product_Code'].nunique() if len(df_inv_other) > 0 else 0:>15,}")
print(f"  Quantity:              {df_inv_other['Quantity'].sum() if len(df_inv_other) > 0 else 0:>15,.0f}")
print(f"  Value:                 £{df_inv_other['Current_Value_GBP'].sum() if len(df_inv_other) > 0 else 0:>14,.2f}")

print(f"\n{'PLANNED RECEIPTS (INBOUND)':<40}")
print(f"  Receipt Count:         {len(df_receipts):>15,}")
print(f"  Total Quantity:        {df_receipts['Quantity'].sum() if len(df_receipts) > 0 else 0:>15,.0f}")

print(f"\n{'PLANNED ISSUES (OUTBOUND)':<40}")
print(f"  Issue Count:           {len(df_issues):>15,}")
print(f"  Total Quantity:        {df_issues['Quantity'].sum() if len(df_issues) > 0 else 0:>15,.0f}")

print(f"\n{'TRANSPORTATION LANES':<40}")
if len(df_tlanes) > 0:
    inbound_lanes = df_tlanes[df_tlanes['Direction'] == 'INBOUND']['Lane_ID'].nunique()
    outbound_lanes = df_tlanes[df_tlanes['Direction'] == 'OUTBOUND']['Lane_ID'].nunique()
    print(f"  Inbound Lanes:         {inbound_lanes:>15,}")
    print(f"  Outbound Lanes:        {outbound_lanes:>15,}")
    print(f"  Total Lane Options:    {len(df_tlanes):>15,}")
else:
    print(f"  No lanes found")

# Net position
current_stock = df_inv_impacted['Quantity'].sum() if len(df_inv_impacted) > 0 else 0
inbound = df_receipts['Quantity'].sum() if len(df_receipts) > 0 else 0
outbound = df_issues['Quantity'].sum() if len(df_issues) > 0 else 0
net_position = current_stock + inbound - outbound

print(f"\n{'NET POSITION AT IMPACTED LOCATION':<40}")
print(f"  Current Stock:         {current_stock:>15,.0f}")
print(f"  + Planned Receipts:    {inbound:>15,.0f}")
print(f"  - Planned Issues:      {outbound:>15,.0f}")
print(f"  = Net Position:        {net_position:>15,.0f}")

print("\n" + "=" * 70)